# ML-06 — Signal Audit & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

EDA on search telemetry and distribution transformations.

## 1. Heavy Skew in Search Exposure

Search impressions and clicks exhibit extreme right-skewed distributions spanning 5 orders of magnitude. We apply `log1p` transformation to ensure numerical stability and prevent outlier domination.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Auto-detect data path: works in local repo or directly in Google Colab
DATA_URL = 'https://raw.githubusercontent.com/AzizullahMemonAi/FlyRank-ML-Assignments/main/data/raw/content_refresh_anonymized.csv'
LOCAL_PATHS = [
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv')
]
data_path = next((p for p in LOCAL_PATHS if p.exists()), None)
df = pd.read_csv(data_path if data_path is not None else DATA_URL)
print('Raw Impressions Quantiles:')
print(df['impressions_90d'].quantile([0.25, 0.5, 0.75, 0.9, 0.99]))
log_imp = np.log1p(df['impressions_90d'])
print('\nLog1p Impressions Summary:')
print(log_imp.describe().round(2))


Raw Impressions Quantiles:
0.25        52.0
0.50       445.0
0.75      3412.0
0.90     21943.5
0.99    389102.3
Name: impressions_90d, dtype: float64

Log1p Impressions Summary:
count    30000.00
mean         6.24
std          2.61
min          0.00
25%          3.97
50%          6.10
75%          8.14
max         16.81
Name: impressions_90d, dtype: float64


## 2. Telemetry Signal Correlations

We audit correlations among key observed metrics: impressions, clicks, position, CTR, engagement, and content age.

In [1]:
clean_pos = df['avg_position'].replace(0, 100)
corr_df = pd.DataFrame({
    'log_impressions': np.log1p(df['impressions_90d']),
    'log_clicks': np.log1p(df['clicks_90d']),
    'clean_position': clean_pos,
    'ctr': df['ctr'].fillna(0),
    'engagement_rate': df['engagement_rate'].fillna(0),
    'scroll_rate': df['scroll_rate'].fillna(0),
    'content_age': df['content_age_days']
}).corr().round(3)
print('Signal Correlation Matrix:')
print(corr_df.to_string())


Signal Correlation Matrix:
                 log_impressions  log_clicks  clean_position    ctr  engagement_rate  scroll_rate  content_age
log_impressions            1.000       0.782          -0.642  0.081            0.124        0.045        0.112
log_clicks                 0.782       1.000          -0.518  0.342            0.215        0.082        0.094
clean_position            -0.642      -0.518           1.000 -0.198           -0.104       -0.032       -0.087
ctr                        0.081       0.342          -0.198  1.000            0.141        0.052        0.021
engagement_rate            0.124       0.215          -0.104  0.141            1.000        0.412        0.015
scroll_rate                0.045       0.082          -0.032  0.052            0.412        1.000        0.008
content_age                0.112       0.094          -0.087  0.021            0.015        0.008        1.000
